### Include Library

In [13]:
from datetime import datetime
import os

# library for cap_f1
from cap_f1 import LLMClient, AtomicProcessor, ResultsRepo
from fewshot_examples import (
    FEWSHOT_DEDUP_MESSAGES,
    FEWSHOT_RECALL_MESSAGES,
    FEWSHOT_PRECISION_MESSAGES,
)

# code for no need for restarting the kernel when python file is updated
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1. build API + processor (inject few-shot examples if you want)
currently fewshot example is at fewshot_examples.py

In [14]:
# 1) build API + processor (inject few-shot examples if you want)
llm = LLMClient()
proc = AtomicProcessor(
    llm,
    fewshot_dedup=FEWSHOT_DEDUP_MESSAGES,  # or None
    fewshot_recall=FEWSHOT_RECALL_MESSAGES,  # or None
    fewshot_precision=FEWSHOT_PRECISION_MESSAGES,  # or None
)

### 2. Load Data

In [15]:
print("Loading caption dataset...")

# number of data points testing
LIMIT = 2

# for filename
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H-%M")

# create folder to save the results
folder_path = f"results/{timestamp}"
os.makedirs(folder_path, exist_ok=True)

# features that we need to extract from the original dataset
org_caption_dataset = ResultsRepo.read_json(
    "../../../data/study-2-output/final-evaluated-captions/low-quality_evaluation_5432-images_2025-04-10_15:29.json"
)
org_caption_dataset = org_caption_dataset[:LIMIT]
org_caption_dataset[0]

Loading caption dataset...


{'image_id': 1,
 'file_name': 'VizWiz_train_00000001.jpg',
 'vizwiz_url': 'https://vizwiz.cs.colorado.edu/VizWiz_visualization_img/VizWiz_train_00000001.jpg',
 'text_detected': True,
 'unrecognizable': 0,
 'framing': 0,
 'blur': 5,
 'obstruction': 0,
 'rotation': 0,
 'too dark': 0,
 'too bright': 0,
 'other': 0,
 'no issue': 0,
 'human_captions': [{'caption': 'A can of Coca Cola on a counter is shown for when one can use a nice, cold drink.',
   'is_precanned': False,
   'is_rejected': False},
  {'caption': 'A black can of Coca Cola Zero calorie soda is on the counter near the coffee maker.',
   'is_precanned': False,
   'is_rejected': False},
  {'caption': 'A kitchen counter the various items on top including a can of Coca-Cola, metal containers, and a teapot.',
   'is_precanned': False,
   'is_rejected': False},
  {'caption': 'a black tin of Coca Cola placed on a black surface',
   'is_precanned': False,
   'is_rejected': False},
  {'caption': 'Black counter with canisters, kettle an

### 3. generate atomics

In [16]:
# 3) generate atomics
print("Generating atomic statements using gpt-4o...")
T_atomics, g_atomics, parsed_T = proc.generate_atomic_statement(
    org_caption_dataset, limit=LIMIT
)

# 3.1) save intermediate
print("Saving intermediate results...")
all_human_captions = []
for item in org_caption_dataset:
    # Filter out human captions that are mention quality issues
    human_captions = [
        hc["caption"]
        for hc in item["human_captions"]
        if hc["caption"] != "Quality issues are too severe to recognize visual content."
    ]
    all_human_captions.append(human_captions)
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/intermediate_{timestamp}.json",
    org_dataset=org_caption_dataset,
    T_atomics=T_atomics,
    g_atomics=g_atomics,
    parsed_T=parsed_T,
    T_org=all_human_captions,
    limit=LIMIT,
)

Generating atomic statements using gpt-4o...


100%|██████████| 2/2 [00:24<00:00, 12.10s/it]

Saving intermediate results...
Saved JSON to: results/2025-08-20_22-47/intermediate_2025-08-20_22-47.json


### 4. evaluate and get recall and precision
- match human caption to model caption
- create recall and precision data

In [17]:
# before calculating F1 score, match sentences between human generated and model generated
print("Evaluating atomic statements...")
eval_out = proc.evaluate_matching(all_human_captions, T_atomics, g_atomics)

# 4.1) save evaluation results
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/eval_{timestamp}.json",
    update_existing=f"{folder_path}/intermediate_{timestamp}.json",
    metadata=eval_out,
    limit=LIMIT,
)

Evaluating atomic statements...


 50%|█████     | 1/2 [00:16<00:16, 16.88s/it]

[gpt-4o-2024-08-06] Recall mismatch: len T=16 vs TP+FN=16


100%|██████████| 2/2 [00:39<00:00, 19.64s/it]

Saved JSON to: results/2025-08-20_22-47/eval_2025-08-20_22-47.json


In [18]:
eval_out

[[{'model_name': 'gpt-4o-2024-08-06',
   'recall': {'TPs': ['There is a can of Coca Cola Zero calorie soda.',
     'The can is on a counter.',
     'A kitchen counter is present.',
     'Metal containers are on the kitchen counter.',
     'There are canisters on the counter.',
     'There is a kettle on the counter.'],
    'FNs': ['The can is black.',
     'There is a coffee maker near the can.',
     'The counter is black.',
     'A teapot is on the kitchen counter.'],
    'Match': [{'T_atomic': 'There is a can of Coca Cola Zero calorie soda.',
      'g_atomic': 'There is a can of Coca-Cola Zero.'},
     {'T_atomic': 'The can is on a counter.',
      'g_atomic': 'The can of Coca-Cola Zero is on a kitchen countertop.'},
     {'T_atomic': 'A kitchen counter is present.',
      'g_atomic': 'The can of Coca-Cola Zero is on a kitchen countertop.'},
     {'T_atomic': 'Metal containers are on the kitchen counter.',
      'g_atomic': 'There are three silver canisters.'},
     {'T_atomic': 'Th

### 5. calculate cap f1 score


In [19]:
# 5) calculate cap f1 score
cap_scores = proc.calculate_cap_f1(eval_out)

# 5.1) save cap f1 score results
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/final_{timestamp}.json",
    update_existing=f"{folder_path}/eval_{timestamp}.json",
    evaluations=cap_scores,
    limit=LIMIT,
)

100%|██████████| 2/2 [00:00<00:00, 54120.05it/s]

Saved JSON to: results/2025-08-20_22-47/final_2025-08-20_22-47.json


In [20]:
cap_scores

[[{'model_name': 'gpt-4o-2024-08-06',
   'recall': 0.6,
   'precision': 0.2222222222222222,
   'cap_f1': 0.32432432432432434},
  {'model_name': 'Llama-3.2-11B-Vision-Instruct',
   'recall': 0.5,
   'precision': 0.3333333333333333,
   'cap_f1': 0.4},
  {'model_name': 'Molmo-7B-O-0924',
   'recall': 0.4,
   'precision': 0.42857142857142855,
   'cap_f1': 0.4137931034482759}],
 [{'model_name': 'gpt-4o-2024-08-06',
   'recall': 0.625,
   'precision': 0.5,
   'cap_f1': 0.5555555555555556},
  {'model_name': 'Llama-3.2-11B-Vision-Instruct',
   'recall': 0.5625,
   'precision': 0.6666666666666666,
   'cap_f1': 0.6101694915254238},
  {'model_name': 'Molmo-7B-O-0924',
   'recall': 0.625,
   'precision': 0.6666666666666666,
   'cap_f1': 0.6451612903225806}]]

In [7]:
# 6) Final JSON → CSV
print("Saving final results into csv...")
ResultsRepo.export_final_csv(
    json_path=f"{folder_path}/final_{timestamp}.json",
    csv_path=f"{folder_path}/final_{timestamp}.csv",
    # model_keys={"gpt":"gpt-4o-2024-08-06", "molmo":"Molmo-7B-O-0924", "llama":"Llama-3.2-11B-Vision-Instruct"}
)

Saving final results into csv...
CSV file saved to: results/2025-08-10_01-41/final_2025-08-10_01-41.csv
